# Train on 221 EMNLP Videos
Features: WavLM 768-dim + prosody 23-dim = 791 total
Labels: EMNLP word-level BIO tags

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import os, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = BASE + '/features_221'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/all'

print('Setup complete')


In [ ]:
# Load all features + labels
feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')])
print('Feature files:', len(feat_files))

X_list, y_list, vids = [], [], []
for f in feat_files:
    vid = f.replace('_features.npy', '')
    label_path = LABEL_DIR + '/' + vid + '.csv'
    if not os.path.exists(label_path):
        print('MISSING:', vid)
        continue
    
    feats = np.load(FEAT_DIR + '/' + f)  # (n_chunks, 791)
    labels_df = pd.read_csv(label_path)
    
    # Map chunks to words using timestamp overlap
    # Each 5s chunk covers ~50 words on average
    n_chunks = len(feats)
    words_per_chunk = max(1, len(labels_df) // n_chunks)
    
    # Aggregate labels per chunk: majority vote
    chunk_labels = []
    for i in range(n_chunks):
        start_word = i * words_per_chunk
        end_word = min((i+1) * words_per_chunk, len(labels_df))
        chunk_df = labels_df.iloc[start_word:end_word]
        # Laugh if any B/I/L tag in this chunk
        is_laugh = any(str(l).strip() in ['B', 'I', 'L'] for l in chunk_df['label'])
        chunk_labels.append(1 if is_laugh else 0)
    
    X_list.append(feats)
    y_list.append(np.array(chunk_labels))
    vids.extend([vid] * n_chunks)

X = np.vstack(X_list)
y = np.concatenate(y_list)
groups = np.array(vids)

print('X:', X.shape, 'y:', y.shape, 'pos rate:', y.mean().round(3))
print('Unique videos:', len(set(vids)))


In [ ]:
# Model + Training
class WordModel(nn.Module):
    def __init__(self, in_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

gkf = GroupKFold(n_splits=5)
fold_f1s = []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups)):
    print('Fold', fold+1)
    Xtr, Xte = X[tr_idx], X[te_idx]
    ytr, yte = y[tr_idx], y[te_idx]
    
    pos_rate = ytr.mean()
    pos_weight = min((1-pos_rate)/pos_rate, 3.0) if pos_rate > 0.01 else 1.0
    
    model = WordModel().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    criterion = nn.BCELoss()
    
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32).to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1).to(device)
    
    best_f1, patience, no_imp = 0, 5, 0
    for ep in range(50):
        model.train()
        for i in range(0, len(Xtr_t), 256):
            bx, by = Xtr_t[i:i+256], ytr_t[i:i+256]
            opt.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            opt.step()
        
        model.eval()
        with torch.no_grad():
            probs = model(torch.tensor(Xte, dtype=torch.float32).to(device)).cpu().numpy().squeeze()
            f = f1_score(yte, (probs>=0.5).astype(int))
            if f > best_f1: best_f1 = f; no_imp = 0
            else: no_imp += 1
            if no_imp >= patience: break
    
    model.eval()
    with torch.no_grad():
        probs = model(torch.tensor(Xte, dtype=torch.float32).to(device)).cpu().numpy().squeeze()
        p = precision_score(yte, (probs>=0.5).astype(int), zero_division=0)
        r = recall_score(yte, (probs>=0.5).astype(int), zero_division=0)
        f = f1_score(yte, (probs>=0.5).astype(int), zero_division=0)
    print(f'  F1={f:.4f} P={p:.4f} R={r:.4f} (pos_rate={pos_rate:.3f}, pos_weight={pos_weight:.2f})')
    fold_f1s.append(f)

print('')
print('CV F1:', np.mean(fold_f1s).round(4), '+/-', np.std(fold_f1s).round(4))

# Save
best_idx = np.argmax(fold_f1s)
torch.save(model.state_dict(), BASE + '/wordlevel_221_model.pt')
print('Model saved')
